# 04 — Disengagement Feature Engineering (v2)
SpiriCom · Huawei Technologies Tunisia · PFE 2026

Consumes `churn_labelled_v6.parquet` + `models/leakage_blocklist.json` from 03b.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 60)
PROC_DIR  = Path('data/processed')
MODEL_DIR = Path('models')
OUT_DIR   = Path('data/outputs')

churn = pd.read_parquet(PROC_DIR / 'churn_labelled_v6.parquet')
blocklist = json.load(open(MODEL_DIR / 'leakage_blocklist.json'))
BLOCKED = set(blocklist['blocked_features'])
print(f'Loaded: {churn.shape[0]:,} rows x {churn.shape[1]} cols')
print(f'Blocklist: {len(BLOCKED)} features')

# NB04-1: labelled population only
data = churn.dropna(subset=['churn']).copy()
data['churn'] = data['churn'].astype(int)
print(f'Labelled rows: {len(data):,} '
      f'(disengaged={int(data["churn"].sum()):,}, '
      f'rate={data["churn"].mean()*100:.1f}%)')


Loaded: 4,896 rows x 34 cols
Blocklist: 31 features
Labelled rows: 2,566 (disengaged=868, rate=33.8%)


## 1 — De-imputation of QoS columns (NB04-2)

In [2]:
# Columns whose *_imputed flag exists in the table: restore NaN where flagged
imp_flag_cols = [c for c in data.columns if c.endswith('_imputed')]
restored = []
for flag in imp_flag_cols:
    base = flag[:-len('_imputed')]
    if base in data.columns and base not in BLOCKED:
        n = int((data[flag] == 1).sum())
        data.loc[data[flag] == 1, base] = np.nan
        restored.append((base, n))
print('Median values removed (back to honest NaN):')
for base, n in restored:
    print(f'  {base:<28s}: {n:>5,} rows -> NaN '
          f'({data[base].isna().mean()*100:.1f}% missing now)')
print('\nImputed flags kept as features (missingness can be signal):')
print([f for f in imp_flag_cols])


Median values removed (back to honest NaN):
  e2e_delay_ms                :   315 rows -> NaN (12.3% missing now)
  client_rtt_ms               :   321 rows -> NaN (12.5% missing now)

Imputed flags kept as features (missingness can be signal):
['dou_total_imputed', 'duration_imputed', 'traffic_5g_imputed', 'traffic_4g_imputed', 'traffic_3g_imputed', 'traffic_2g_imputed', 'e2e_delay_ms_imputed', 'client_rtt_ms_imputed']


## 2 — Device & capability features (NB04-5)

In [3]:
gen = data['generation'].fillna('UNKNOWN').astype(str).str.upper()
data['is_2g_only'] = (gen == '2G').astype(int)
data['has_3g']     = gen.str.contains('3G').astype(int)
data['has_lte']    = gen.str.contains('LTE').astype(int)
data['has_nr']     = gen.str.contains('NR').astype(int)
print('Capability flags vs disengagement rate:')
for f in ['is_2g_only', 'has_3g', 'has_lte', 'has_nr']:
    g = data.groupby(f)['churn'].agg(['mean', 'count'])
    print(f'\n{f}:')
    print((g['mean'] * 100).round(1).astype(str) + '%  (n=' +
          g['count'].astype(str) + ')')

# NB04-FIX-2: detect perfect complement between has_3g and is_2g_only.
# In this dataset every non-2G-only device has 3G capability, so
# has_3g == 1 - is_2g_only for every row. has_3g is computed here
# for inspection but EXCLUDED from FLAG_FEATURES in the assembly
# step (cell 9) to avoid multicollinearity in LR and misleading SHAP.
_is_complement = (data['has_3g'] == 1 - data['is_2g_only']).all()
if _is_complement:
    print('\nNB04-FIX-2: has_3g == 1 - is_2g_only (perfect complement '
          'confirmed) — excluded from feature matrix.')

# Brand: top 10 + OTHER, one-hot
top_brands = data['brand'].value_counts().head(10).index
data['brand_grp'] = np.where(data['brand'].isin(top_brands),
                             data['brand'], 'OTHER')
brand_oh = pd.get_dummies(data['brand_grp'], prefix='brand', dtype=int)
print(f'\nBrand one-hot: {list(brand_oh.columns)}')


Capability flags vs disengagement rate:

is_2g_only:
is_2g_only
0    30.2%  (n=2419)
1     93.2%  (n=147)
dtype: object

has_3g:
has_3g
0     93.2%  (n=147)
1    30.2%  (n=2419)
dtype: object

has_lte:
has_lte
0     85.6%  (n=174)
1    30.1%  (n=2392)
dtype: object

has_nr:
has_nr
0    34.9%  (n=2146)
1     28.3%  (n=420)
dtype: object

NB04-FIX-2: has_3g == 1 - is_2g_only (perfect complement confirmed) — excluded from feature matrix.

Brand one-hot: ['brand_APPLE', 'brand_HONOR', 'brand_HUAWEI', 'brand_INFINIX', 'brand_ITEL', 'brand_OPPO', 'brand_OTHER', 'brand_REALME', 'brand_SAMSUNG', 'brand_TECNO', 'brand_XIAOMI']


## 3 — Geography & profile features

In [4]:
# layer2name: top 12 zones + OTHER (24 zones total, long tail)
oh_frames = [brand_oh]
if 'layer2name' in data.columns:
    top_zones = data['layer2name'].value_counts().head(12).index
    data['zone_grp'] = np.where(data['layer2name'].isin(top_zones),
                                data['layer2name'], 'OTHER')
    oh_frames.append(pd.get_dummies(data['zone_grp'], prefix='zone', dtype=int))

# Low-cardinality profile categoricals
for col in ['mobility_class', 'usertype', 'user_class', 'tertype']:
    if col in data.columns:
        vals = data[col].fillna('UNKNOWN').astype(str).str.upper()
        if vals.nunique() <= 10:
            oh_frames.append(pd.get_dummies(vals, prefix=col, dtype=int))
            print(f'{col}: {vals.nunique()} categories one-hot encoded')

oh = pd.concat(oh_frames, axis=1).reset_index(drop=True)
# NB04-fix: drop constant one-hots (e.g. usertype with 1 category)
const = [c for c in oh.columns if oh[c].nunique() <= 1]
if const:
    print(f'Dropped constant one-hot columns: {const}')
    oh = oh.drop(columns=const)
print(f'\nTotal one-hot columns: {oh.shape[1]}')


mobility_class: 3 categories one-hot encoded
usertype: 1 categories one-hot encoded
user_class: 7 categories one-hot encoded
tertype: 2 categories one-hot encoded
Dropped constant one-hot columns: ['usertype_DATA USER']

Total one-hot columns: 36


## 4 — Assemble the feature matrix (blocklist enforced)

In [5]:
NUMERIC_CANDIDATES = [c for c in [
    'e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms',
    'client_packet_loss_rate', 'server_packet_loss_rate',
    'page_response_delay', 'page_download_throughput',
    'dns_delay', 'dns_sr', 'tcp_connection_sr',
    'voip_voice_downlink_throughput', 'voip_voice_uplink_throughput',
    'number_of_regions',
] if c in data.columns and c not in BLOCKED]

# NB04-FIX-1: dou_total_imputed and duration_imputed are guaranteed 0
# for every row in the labelled 2,566-customer population. By design in
# NB03b, a customer is only labelled when BOTH dou_total_imputed==0 AND
# duration_imputed==0 (imputed rows get churn=NaN and are excluded
# via dropna above). Including them adds 2 zero-variance features with
# zero predictive power — they would also distort SHAP importance.
LABEL_IMP_FLAGS = {'dou_total_imputed', 'duration_imputed'}

# NB04-FIX-2: has_3g == 1 - is_2g_only (confirmed in cell 5).
# Including both creates a perfect linear dependency. Dropped here.
# Retained capability ladder: is_2g_only (93.2%!) / has_lte / has_nr.
FLAG_FEATURES = (
    [c for c in data.columns
     if c.endswith('_imputed') and c not in LABEL_IMP_FLAGS]
    + ['is_2g_only', 'has_lte', 'has_nr']
)

X = pd.concat([data[NUMERIC_CANDIDATES + FLAG_FEATURES]
                 .reset_index(drop=True),
               oh.reset_index(drop=True)], axis=1)
y = data['churn'].reset_index(drop=True)
msisdn = data['msisdn'].reset_index(drop=True)

# Final guard: nothing from the blocklist may survive
leaks = [c for c in X.columns if c in BLOCKED]
assert not leaks, f'BLOCKED FEATURES IN X: {leaks}'

# NB04-FIX-3: zero-variance safety net — mirrors the one-hot guard in
# cell 7. Catches any constant column that slips through the explicit
# exclusions above (e.g. after a future data refresh).
const_X = [c for c in X.columns if X[c].nunique() <= 1]
if const_X:
    print(f'NB04-FIX-3: dropped constant columns: {const_X}')
    X = X.drop(columns=const_X)
else:
    print('NB04-FIX-3: zero-variance check passed — no constant columns.')

print(f'Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'  numeric QoS    : {len(NUMERIC_CANDIDATES)}')
print(f'  flags          : {len(FLAG_FEATURES)}')
print(f'  one-hot        : {oh.shape[1]}')
print(f'  NaN cells      : {int(X.isna().sum().sum()):,} '
      '(intentional - XGBoost native missing handling)')


NB04-FIX-3: zero-variance check passed — no constant columns.
Feature matrix: 2,566 rows x 56 features
  numeric QoS    : 11
  flags          : 9
  one-hot        : 36
  NaN cells      : 636 (intentional - XGBoost native missing handling)


## 5 — Signal preview
Univariate check before modeling: point-biserial correlation for numeric features,
disengagement rate per one-hot group. Expect **modest** values — the strong volume
signals are blocked by design; what remains is the honest experience/device/geo signal.

In [6]:
print('Numeric features - point-biserial correlation with disengagement:')
rows = []
for c in NUMERIC_CANDIDATES:
    valid = X[c].notna()
    if valid.sum() > 100 and X.loc[valid, c].std() > 0:
        r = np.corrcoef(X.loc[valid, c], y[valid])[0, 1]
        rows.append((c, r, int(valid.sum())))
for c, r, n in sorted(rows, key=lambda t: -abs(t[1])):
    bar = '#' * int(abs(r) * 50)
    print(f'  {c:<34s} r={r:>7.3f}  n={n:>5,}  {bar}')

print('\nTop one-hot groups by disengagement-rate gap vs average '
      f'({y.mean()*100:.1f}%):')
gaps = []
for c in oh.columns:
    m = (oh[c] == 1).values   # NB04-fix: positional mask, index-safe
    if m.sum() >= 30:
        gaps.append((c, y[m].mean() * 100, int(m.sum())))
for c, rate, n in sorted(gaps, key=lambda t: -abs(t[1] - y.mean()*100))[:12]:
    print(f'  {c:<34s} {rate:>5.1f}%  (n={n:,})')


Numeric features - point-biserial correlation with disengagement:
  tcp_connection_sr                  r= -0.523  n=2,566  ##########################
  dns_sr                             r= -0.387  n=2,566  ###################
  number_of_regions                  r= -0.281  n=2,566  ##############
  client_packet_loss_rate            r= -0.113  n=2,566  #####
  client_rtt_ms                      r=  0.098  n=2,245  ####
  dns_delay                          r= -0.095  n=2,566  ####
  server_rtt_ms                      r=  0.075  n=2,566  ###
  page_download_throughput           r= -0.074  n=2,566  ###
  e2e_delay_ms                       r=  0.066  n=2,251  ###
  server_packet_loss_rate            r= -0.066  n=2,566  ###
  page_response_delay                r= -0.048  n=2,566  ##

Top one-hot groups by disengagement-rate gap vs average (33.8%):
  tertype_FEATUREPHONE                97.1%  (n=136)
  user_class_OTHER_TRAFFIC            91.3%  (n=495)
  brand_OTHER                         

## 6 — Persist features + reproducible split (NB04-4)

In [7]:
X_tr, X_te, y_tr, y_te, id_tr, id_te = train_test_split(
    X, y, msisdn, test_size=0.25, stratify=y, random_state=42)

out = X.copy()
out['churn']  = y
out['msisdn'] = msisdn
out['split']  = np.where(out['msisdn'].isin(set(id_te)), 'test', 'train')
feat_path = PROC_DIR / 'churn_features_v6.parquet'
out.to_parquet(feat_path, index=False)

meta = {
    'generated_at'   : datetime.now().isoformat(),
    'label_version'  : 'v6',
    'n_rows'         : int(len(out)),
    'n_features'     : int(X.shape[1]),
    'feature_columns': list(X.columns),
    'numeric_qos'    : NUMERIC_CANDIDATES,
    'flag_features'  : FLAG_FEATURES,
    'split'          : {'train': int((out['split']=='train').sum()),
                        'test' : int((out['split']=='test').sum()),
                        'stratified': True, 'random_state': 42},
    'disengaged_rate': round(float(y.mean()), 4),
    'blocklist_enforced': True,
}
json.dump(meta, open(MODEL_DIR / 'churn_features_meta.json', 'w'), indent=2)
print(f'Saved: {feat_path} ({out.shape[0]:,} x {out.shape[1]})')
print(f'Saved: {MODEL_DIR / "churn_features_meta.json"}')
print(f'Train {meta["split"]["train"]:,} / Test {meta["split"]["test"]:,} '
      f'(stratified, seed 42)')
print('\nNext -> 05_Churn_Modeling: LR / RF / XGBoost on these features.')
print('Honest expectations: with volume features blocked, headline metrics')
print('will be modest; report PR-AUC + lift vs the 33.8% baseline, and let')
print('SHAP (NB06) carry the business insight (2G-only devices, zones, QoS).')


Saved: data\processed\churn_features_v6.parquet (2,566 x 59)
Saved: models\churn_features_meta.json
Train 1,924 / Test 642 (stratified, seed 42)

Next -> 05_Churn_Modeling: LR / RF / XGBoost on these features.
Honest expectations: with volume features blocked, headline metrics
will be modest; report PR-AUC + lift vs the 33.8% baseline, and let
SHAP (NB06) carry the business insight (2G-only devices, zones, QoS).
